# Exploration (sorties effacées avant chaque commit)

Règle : on explore **le pli d'entraînement** (2012-2023), pas le jeu entier. La validation et le test ne servent pas à choisir des variables. Les sorties sont effacées avant le commit (Kernel → Restart & Clear Output).

Prérequis : `make reproduce` a tourné une fois (données téléchargées, coupe écrite).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from energie.config import load_config
from energie.data import TARGET
from energie.split import load_fold
cfg = load_config()   # se résout depuis la racine du dépôt, quel que soit le dossier courant
train = load_fold("train")
train[["date_heure", TARGET, "heure_type_jour", "mois"] + [c for c in train.columns if c.startswith("conso_moins_")]].head()

In [ ]:
# Le cycle de la journée selon le type de jour : la structure que l'OLS apprend avec heure_type_jour.
profil = train.assign(heure=train["heure_type_jour"].str[:2].astype(int), type_jour=train["heure_type_jour"].str[3:])
profil.pivot_table(index="heure", columns="type_jour", values=TARGET, aggfunc="mean").round(0).plot(title="Consommation moyenne par heure (MW), 2012-2023");

In [ ]:
# La saison : la consommation d'hiver dépend du chauffage électrique, donc de la température (à ajouter).
train.groupby("mois")[TARGET].mean().round(0).plot(kind="bar", title="Consommation moyenne par mois (MW), 2012-2023");

In [ ]:
# La tendance par année : la baisse depuis 2022 (crise énergétique, sobriété) change le niveau à prévoir.
train.groupby("annee")[TARGET].mean().round(0)